# Step 22.5 — Flash-style attention：分块以后，怎样仍然算同一个加权平均

## 独立工作副本（以下命令本次未执行）

在仓库根目录同步母版：

```bash
git pull
```

再复制到已有的 `notebooks/work/` 目录；时间戳区分副本，`cp -n` 在同名文件存在时不覆盖笔记：

```bash
cp -n notebooks/101_step22_5_flash_style_attention.ipynb notebooks/work/101_step22_5_flash_style_attention_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

本节先纸算，再读源码，最后列未来可主动运行的观察。它不需要读取语料、模型或 checkpoint。MLX 接口固定为 **v0.32.2**。

**材料状态：**教材历史内容修正 0/2＋本次教学修订 1 次，累计 **1/2**。所有代码单元 `execution_count=null`、`outputs=[]`；本次仅静态审阅，没有执行 Notebook、MLX、测试、训练、验证、实验、绘图或测量，也没有生成真实运行结果。


## 1. 从上一节的 dense score 中取出一行

[22.4 的 reference](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/f476c9d071635b95766f53c0e55be3cca4e61148/notebooks/100_step22_4_mqa_gqa_attention.ipynb)先形成 score，再做 softmax，最后乘 V。本节不改变 query/KV head 的分组，而是问：**同一行的 key 太多时，能否分块累计，最后仍得到同一个输出？**

先固定一个 batch、一个 head 和一个 query。令该 query 对第 j 个 key 的已缩放分数为 \(s_j=q^\top k_j/\sqrt d\)，其 value 为向量 \(v_j\)。没有 mask 时：

$$
p_j=\frac{e^{s_j}}{\sum_i e^{s_i}},
\qquad o=\sum_j p_jv_j.
$$

复用已学的 stable softmax：所有指数都减去同一个最大值 m，分子、分母同时乘 \(e^{-m}\)，比值不变。因而可以先积累**未归一化**的量，最后才相除：

$$
o=\frac{\sum_j e^{s_j-m}v_j}{\sum_j e^{s_j-m}},
\qquad m=\max_j s_j.
$$

这里的“分块”沿 **key/token 轴**；每个 value 的所有分量一起参与。下面的数学 reference 假设输入非空、分数和 value 有限，暂不加入 causal/padding mask。


## 2. 三个状态各记什么，为什么必须共同重标定

处理完一部分位置 S 后，保存以下状态（running state）：

| 符号 | 含义 | 单行 shape | 源码变量 |
|---|---|---|---|
| m | 已处理分数的最大值 | 标量 `()` | `m` |
| l | \(\sum_{j\in S}e^{s_j-m}\)，未归一化权重的总量 | 标量 `()` | `total` |
| u | \(\sum_{j\in S}e^{s_j-m}v_j\)，未归一化加权和 | `(dv,)` | `weighted` |

u 是向量，l 是标量；u 还没有除以 l。新块 b 到来后，需要统一使用新的最大值：

$$
m'=\max(m,\max s_b),\qquad \alpha=e^{m-m'}.
$$

对任何旧位置，\(e^{s_j-m'}=e^{s_j-m}e^{m-m'}=\alpha e^{s_j-m}\)。这正是旧累计要换基准的原因：

$$
\boxed{
l'=\alpha l+\sum_{j\in b}e^{s_j-m'},\qquad
u'=\alpha u+\sum_{j\in b}e^{s_j-m'}v_j
}
$$

若新最大值升高，则 \(0<\alpha<1\)；若没有升高，则 \(\alpha=1\)。**旧 l 乘 α，旧 u 的每个分量也乘 α**，然后各自加入新块的贡献。不能只缩放其中一个，也不能把旧比值 u/l 乘 α：同时缩放分子分母并不改变旧比值，但会改变旧部分相对于新块的权重总量。只保留旧平均值、丢掉 l，便缺少合并所需的信息。

直到所有块处理完，才返回 \(o=u/l\)。这是一行 softmax 的累计方法，不是把每块独立归一化后的输出直接相加。


## 3. 三块纸算：第一次最大值上升，下一次不变

这是按实数公式推导的例子，不是运行输出。取：

$$
s=[0,\ln2,0],\qquad
V=\begin{bmatrix}2&0\\0&4\\6&2\end{bmatrix}.
$$

`scores` 为 `(3,)`，`values` 为 `(3,2)`；`tile_size=1`，每块分数 `(1,)`、value `(1,2)`，输出应为 `(2,)`。两个 value 分量共用同一套权重。

**初始化：**还没有旧位置，源码设 `m=None`、l=0、u=[0,0]。第一块直接取自身最大值，并令 α=0，表示没有旧贡献；不计算 `None−m'`，也不借助 \(-\infty-(-\infty)\)。

| 处理的块 | 新最大值 m′ | α | 新块权重 \(e^{s_b-m'}\) | 缩放后的旧 l、旧 u | 加入新块后的 l′、u′ |
|---|---|---|---|---|---|
| 1：s=0，v=[2,0] | 0 | 0（初始化分支） | 1 | 0，[0,0] | 1，[2,0] |
| 2：s=ln2，v=[0,4] | ln2 | 1/2 | 1 | 1/2，[1,0] | 3/2，[1,4] |
| 3：s=0，v=[6,2] | ln2 | 1 | 1/2 | 3/2，[1,4] | 2，[4,5] |

第二块最关键：最大值从 0 升到 ln2，旧分母 \(1\to1/2\)，旧分子 \([2,0]\to[1,0]\)。第三块只增加 \((1/2)[6,2]=[3,1]\)，因此最终：

$$
o=[4,5]/2=[2,5/2].
$$

直接核对：未减最大值的指数权重为 [1,2,1]，总和为 4；分子为 \([2,0]+2[0,4]+[6,2]=[8,10]\)，除以 4 仍为 [2,5/2]。这是同一数学目标；浮点运算顺序改变时，不要求逐位相同。


### 一个具体错误：换了最大值，却原样保留旧累计

仍看第二块。若错误地把旧 l=1、u=[2,0] 直接加上新块按 m′=ln2 算出的贡献，会得到：

$$
l_{\mathrm{wrong}}=1+1=2,\qquad
u_{\mathrm{wrong}}=[2,0]+[0,4]=[2,4],
\qquad o_{\mathrm{wrong}}=[1,2].
$$

正确的前两块输出是 \([1,4]/(3/2)=[2/3,8/3]\)。错误来自混用了两个指数基准：旧项按 m=0 计重，新项却按 m′=ln2 计重。后面的最终除法不会自动修复这个错误。


## 4. 把纸算逐行对回 helper

下方 `online_attention_row` 接收**已经算好的**一行 scores，不负责生成 QK 点积。设当前块实际长度为 b（最后一块可以短于 `tile_size`）：

| 源码步骤 | shape / 对应数学 |
|---|---|
| `sb = scores[start:start + tile_size]` | `(b,)`；连续取一组 key 分数 |
| `vb = values[start:start + tile_size]` | `(b,dv)`；同一组 key 的 value |
| `block_max = mx.max(sb)` | 标量；只对这一块求最大值 |
| `new_m = ...`、`alpha = ...` | 统一新旧基准；首块用 `m is None` 特判 |
| `weights = mx.exp(sb - new_m)` | `(b,)`；标量广播到块内每个位置 |
| `weights[:, None] * vb` | `(b,1) × (b,dv) → (b,dv)`；这是逐元素乘法 |
| `mx.sum(..., axis=0)` | 沿块内 key 轴相加，保留 `(dv,)` |

真正实现共同重标定的是这两行，不能只保留其中一行的 α：

```text
total = alpha * total + mx.sum(weights)
weighted = alpha * weighted + mx.sum(weights[:, None] * vb, axis=0)
```

随后 `m = new_m` 保存下一轮要用的基准；循环结束的 `weighted / total` 才做归一化。源码内部把每块转成 float32，这是本 reference 的数值选择，不是任意精度的保证。下面保留原 helper，未执行。


In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def online_attention_row(scores, values, tile_size):
    if scores.ndim != 1 or values.ndim != 2 or values.shape[0] != scores.shape[0]:
        raise ValueError("expect scores (T,) and values (T,dv)")
    if scores.shape[0] == 0 or type(tile_size) is not int or tile_size <= 0:
        raise ValueError("nonempty input and positive integer tile_size required")
    # 教学 reference：所有分数/values 有限；没有 causal 或 padding mask。
    m = None
    total = mx.array(0., dtype=mx.float32)
    weighted = mx.zeros((values.shape[1],), dtype=mx.float32)
    for start in range(0, scores.shape[0], tile_size):
        sb = scores[start:start + tile_size].astype(mx.float32)
        vb = values[start:start + tile_size].astype(mx.float32)
        block_max = mx.max(sb)
        new_m = block_max if m is None else mx.maximum(m, block_max)
        alpha = mx.array(0.) if m is None else mx.exp(m - new_m)
        weights = mx.exp(sb - new_m)
        total = alpha * total + mx.sum(weights)
        weighted = alpha * weighted + mx.sum(weights[:, None] * vb, axis=0)
        m = new_m
    return weighted / total



### 输入检查实际做到了哪里

- 显式检查一维 scores、二维 values、相同的 T，并拒绝 T=0。`tile_size` 必须是正的 Python `int`；`type(...) is int` 也会拒绝布尔值。
- 对 T>0、正步长的 `range(0,T,tile_size)`，每个切片至少含一个位置；允许最后一块不满。它没有“传入空块列表”的接口；若擅自改成可含空块的迭代器，`mx.max(sb)` 的前提就变了。
- “分数和 values 有限”目前是**调用前提和注释**，不是 `isfinite` 防御检查。float32 转换也可能把过大值变成 Inf；有限输入、稳定指数并不保证加权和与后续全部计算永不溢出。非常小的指数权重也可能下溢为零。
- 此 helper 没有 mask 参数，也没有全屏蔽块/行处理协议。若首块分数全为 −Inf，`sb - new_m` 会出现 \(-\infty-(-\infty)\)，继而产生 NaN；首块 α=0 不能救回新块的 NaN。若此前 m 已有限，后续全 −Inf 块可以贡献零权重，但这不等于该函数已经支持通用 mask。整行全被屏蔽时分母没有有效权重，超出本 reference 的定义范围。

这些是阅读现有实现得到的边界，不是运行检查结论。先保持“非空、有限、无 mask”的单行推导，再单独看后面的等长 causal 示例。


## 5. Python reference、分块算法、fused kernel 是三个层次

| 层次 | 本节怎样呈现 | 不能据此推出什么 |
|---|---|---|
| Python 单行 reference | 已有 `scores (T,)` 和 `values (T,dv)`，用 Python 循环说明 m/l/u 的累计 | 不是 GPU kernel；完整 scores 已作为输入存在，MLX lazy graph 可能保留各块节点 |
| 分块 attention 算法 | 从 Q 与 K/V 的块逐步生成 scores，为多行各保留 m/l/u，避免完整 T×T score/probability 中间量写回较慢内存 | 不等于把 dense attention 改成稀疏或线性算术量 |
| 实际 fused kernel | 在设备上协调点积、归一化、value 累计及局部存储，减少中间读写与独立调度 | Python 代码写成几行、或 API 名称含 fast，都不能证明选中了某个 kernel 或实际更快 |

对本节等长、每头维度 d 的 MHA，普通 scores 和 probabilities 为 `(B,H,T,T)`，输出为 `(B,H,T,d)`。逐行状态是 m/l 标量与 u 长 d 向量，但真正设备程序还要容纳输入、输出、tiles、工作区等；**逻辑状态大小不等于总内存或实测峰值**。

保持 dense 计算目标时，主要算术规模仍为 \(O(BHT^2d)\)。分块主要改变存储/读写与调度方式，不意味着所有打分都省掉。FlashAttention 的 IO-aware tiling 是这一思想的代表，参见[原论文](https://arxiv.org/abs/2205.14135)；这里没有移植或测量其硬件实现。


## 6. 另一个对照：等长 causal dense reference

下面是与 fast API 比较的**另一条源码路径**，不是给上面的 online helper 悄悄加上 mask。它接收相同 head 数、相同序列长度的 Q/K/V，显式形成完整 score：

`(B,H,T,d) @ (B,H,d,T) → (B,H,T,T)`，再乘 \(d^{-1/2}\)。`allowed[i,j] = (i >= j)` 的 `(T,T)` 布尔 mask 会广播到 batch/head 轴，只允许 query i 读取 key 0…i。每一行至少保留自身位置，所以等长非空情形不会出现全屏蔽行。

未来观察使用 `q/k/v: (1,2,3,4)`；score 是 `(1,2,3,3)`，mask 是 `(3,3)`，输出是 `(1,2,3,4)`，scale 为 \(4^{-1/2}=0.5\)。函数没有提供完整的通用 shape 验证；“等长 MHA”是这里的调用契约，不是增量 cache 或 GQA 实现。


In [ ]:
def dense_causal_attention(q, k, v):
    # Equal-length MHA only，非增量 cache 示例。
    T = q.shape[-2]
    scores = (q @ k.transpose(0, 1, 3, 2)) * (q.shape[-1] ** -0.5)
    allowed = mx.arange(T)[:, None] >= mx.arange(T)[None, :]
    return mx.softmax(mx.where(allowed, scores, float("-inf")), axis=-1) @ v

### 固定 MLX v0.32.2：实际调用了什么

未来单元的调用是 `mx.fast.scaled_dot_product_attention(q, k, v, scale=0.5, mask="causal")`。以[固定版本绑定源码](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)为准：

- `scale` 是必须以 keyword 传入的参数；本例 0.5 与 dense reference 的 \(d^{-1/2}\) 相同。
- `"causal"` 使用 lower-right alignment（最后 query 对齐最后 key）。本例 Tq=Tkv=3，恰好等于上面的下三角；不要把这个对照结论搬到不等长的 decode/prefill 输入。
- 若使用 array mask，它必须能广播到 score shape，且遵守 Boolean/additive mask 的类型契约；本节只使用字符串 causal。
- 默认 `force_fused=False`，没有强迫特定后端。强迫 fused 可能遇到不支持的配置或更慢的路径，本课没有这样设置。

源码说明 softmax 采用 float32；这不意味着任意设备、输入精度或执行路径都与 dense reference 逐位一致。fast API 是接口承诺，不是已经选用某一篇 FlashAttention 论文实现、已经加速或降低实际峰值的证据。


## 7. 未来观察：两个比较，分别回答一个问题

下面保留原观察源码，本次不调用。未来在独立工作副本中主动执行时，先预测每个输出 shape，再分开看：

1. **单行数学对照：**原例 `scores=[0,ln2]`、`values=[[2],[5]]`，比较 online 与 dense，两个输出均应为 `(1,)`。它是二位置、单分量例子，不是上文三块二维纸算的运行记录。
2. **causal 接口对照：**固定 `(1,2,3,4)` 的 Q/K/V，比较完整 dense 与 fast 输出；本单元没有随机数，也没有抽样或概率验证。

判断问题：即使第二项未来得到很小的最大绝对差，能否据此认定第一项 Python 循环具有 fused kernel 的峰值内存和速度？先写清“计算对象是否相同、实际执行路径是什么、缺了哪类测量”，再决定未来是否另做性能实验。本母版不填写实测数值。


In [ ]:
scores = mx.array([0., math.log(2)], dtype=mx.float32)
values = mx.array([[2.], [5.]], dtype=mx.float32)
online = online_attention_row(scores, values, tile_size=1)
dense = mx.softmax(scores, axis=-1) @ values
mx.eval(online, dense)
print("single-row outputs:", online.tolist(), dense.tolist())

q = mx.arange(24, dtype=mx.float32).reshape(1, 2, 3, 4) / 24
k, v = q + 0.1, q * 2
reference = dense_causal_attention(q, k, v)
fast = mx.fast.scaled_dot_product_attention(q, k, v, scale=0.5, mask="causal")
error = mx.max(mx.abs(reference - fast))
mx.eval(reference, fast, error)
print("future observed error:", error.item())

### 如何读将来的数字，而不多下结论

`error = mx.max(mx.abs(reference - fast))` 会对所有 batch/head/query/输出分量取一个最大绝对差，是标量。它只能概括这一组输入的数值差异；不是相对误差、误差的平均值、所有输入正确性的证明或性能指标。

若差异不符合预期，先核对 Q/K/V、scale、mask 对齐、shape、dtype，再考虑浮点计算顺序。这里没有预设“必为 0”的断言，也没有已观察的差值。`mx.eval` 是未来显式求值边界，不是已做计时；代码没有记录 latency 或峰值内存。公平的性能比较还需另定设备、预热、同步边界与 allocator 统计范围。


## 8. 可选扩展与基线边界

本节改变的是 attention 的执行思路。此前 weight tying、RoPE、SwiGLU、GQA 涉及可选架构变化；fast SDPA 是可选执行替换；KV cache 则是额外的推理状态设计。它们不是本节纸算成立的前提，也不能由一次数值对照宣称已完成集成。

保留已经验收的最小模型作为 baseline。未来 Step 23 即使整理代码结构，也不能悄悄混入这些扩展、改变旧 checkpoint 的架构含义。本节只到概念和源码对照，不交付下一节实现。


## 9. 练习（不附答案）

1. 把三项 scores 分成两块，逐步列出 m/l/u 的更新。
2. 说明为什么每块分别 softmax 后直接把输出相加不是完整 softmax。
3. 设计未来公平比较 dense 与 fast 的记录表字段，区分理论复杂度、实际显存和延迟，不填写虚构数值。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。


## 10. 阅读来源与完成边界

- [MLX v0.32.2 fast Python 绑定与接口说明](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)
- [MLX v0.32.2 attention 执行与 fallback 源码](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/fast.cpp)
- [FlashAttention 原论文：IO-aware exact attention](https://arxiv.org/abs/2205.14135)

本次只修订教学文本并分开两个已有函数的代码单元；保留原函数、观察源码及行为。所有单元均未执行，无输出。教材内容修订累计 **1/2**；它与旧教学对话的回复交付故障记录分开，不重置其他历史计数。

材料讲清了 m/l/u 的含义、重标定和实现边界，不等于读者已经掌握，也不等于运行、性能或训练验证。此轮停在 22.5。
